In [0]:
%run "../includes/librerias"

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
#Definimos parametros
v_archivo = "person"
v_esquema = "movie_silver"
v_tabla = "persons"
v_partition = "file_date"
dbutils.widgets.text("p_esquema", v_esquema)
dbutils.widgets.text("p_tabla", v_tabla)

In [0]:
#1. leemos el archivo JSON usando "DataFrameReader" de Spark

# Define el la estructura personName
name_schema = StructType(fields = [
    StructField("forename", StringType(), True),
    StructField("surname", StringType(), True)
])

# Define la estrucutura principal
person_schema = StructType([
    StructField("personId", IntegerType(), False),
    StructField("personName", name_schema)        ##Se utiliza el esquema definido anteriormente
])


# Cargamos el archivo utilizando la estructura definida

person_df = spark.read\
    .schema(person_schema)\
    .json(f"{bronze_folder_path}/{v_file_date}/{v_archivo}.json")

# Mostramos el resultado
display(person_df)

In [0]:
#Paso 2 - Renombrar, añadir y dar formato a las columnas requeridas
person_with_columns_df = person_df\
    .withColumnRenamed("personId", "person_id")\
    .withColumn("name", 
                concat(
                    col("personName.forename"),
                    lit(" "),
                    col("personName.surname")
                      )
                )
    
person_with_columns_df = person_with_columns_df.select(col("person_id"), col("name"))


person_with_columns_df = add_ingestion_date(person_with_columns_df)
person_with_columns_df = add_env(person_with_columns_df)
person_with_columns_df = add_file_date (person_with_columns_df)
display(person_with_columns_df)

In [0]:
#Paso 5 - borramos la informacion de la tabla antes de cargarla
overwrite_partition (v_esquema, v_tabla, v_partition, v_file_date)

In [0]:
#Paso 4 - Guardar datos en datalake en formato parket
person_with_columns_df.write.mode("append").partitionBy(v_partition).format("delta").saveAsTable(f"{v_esquema}.{v_tabla}")
print(f"Se insertaron {person_with_columns_df.count()} registros en la tabla {v_esquema}.{v_tabla}")


In [0]:
dbutils.notebook.exit("El notebook 05. Ingestion File person, termino correctamente")